In [ ]:
import json
import pandas as pd
from tqdm import tqdm
# row 생략 없이 출력
pd.set_option('display.max_rows', None)
# col 생략 없이 출력
pd.set_option('display.max_columns', None)

from pathlib import Path

# 노트북을 어느 하위 폴더에서 실행해도 저장소 루트를 찾습니다.
_current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (_current_dir, *_current_dir.parents)
    if (path / "notebooks").is_dir() and (path / "data").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"


파일에는 총 1000개씩 블록이 저장되어 있고, 블록에는 event의 개수에 따라서 크기가 좀 천차만별이다. 여기서 event id값을 우선 뽑아 본 다음에 우리가 필요한 event만 뽑아서 정리해보자. 우리가 원하는 것은 token의 이동과 관련되어 있다. 
- withdraw

- transfer → balances.Transfer, balances.Endowed

- deposit → staking.Rewarded, staking.PayoutStarted

- locked → balances.Locked

이렇게 정리해달라고 하시는데... 뭔가 더있는지 확인해봐야 할 것 같다.

In [ ]:
input_dir = DATA_DIR / "raw" / "avail" / "blocks"
output_dir = DATA_DIR / "interim" / "avail" / "events"
output_dir.mkdir(parents=True, exist_ok=True)

input_paths = sorted(
    input_dir.glob("AVAIL_blocks_*.json"),
    key=lambda path: int(path.stem.split("_")[2]),
)

for json_path in tqdm(input_paths, desc="Overall Progress", unit="file"):
    with json_path.open() as file:
        blocks = json.load(file)

    event_frames = []
    for block in tqdm(blocks, desc=json_path.stem, leave=False):
        block_data = block['data']
        if not block_data['events']:
            continue

        events = pd.DataFrame(block_data['events'])
        events['timestamp'] = block_data['block_timestamp']
        event_frames.append(events)

    events = pd.concat(event_frames, ignore_index=True) if event_frames else pd.DataFrame()
    range_suffix = json_path.stem[len("AVAIL_blocks_"):]
    events.to_csv(output_dir / f"AVAIL_events_{range_suffix}.csv", index=False)

print(f"Processing complete. Total files processed: {len(input_paths)}")
